## Notebook — Sync Divalto C8 → Supabase (`supplier_accounting_entries`)

### Objectif
Alimenter la table Supabase `supplier_accounting_entries` à partir de `lakehouse_gold.C8_gold` — toutes les écritures comptables sur **comptes auxiliaires F*** (OD, banque, à-nouveau, factures Gescom incluses).

C'est l'envers de la sync Gescom : on remonte TOUTES les écritures fournisseur, et l'App Task se charge ensuite de rattacher celles **hors Gescom** (= sans CF/FF) à des `it_budget_lines`. Cf. brief produit complet dans la conversation.

### Source
- `lakehouse_gold.C8_gold` — écritures comptables grain ligne (1 ligne = 1 `(DOS, JNL, ECRNO, ECRLG)`).

### Destination Supabase
- `supplier_accounting_entries` — clé d'idempotence : `entry_key = md5(DOS|JNL|ECRNO|ECRLG)`.

### Filtres appliqués
- `compte LIKE 'F%'` (auxiliaire fournisseur Divalto).
- `ecrdt >= 2022-01-01` (cf. brief §9.3 — ajustable).

### Heuristique `has_gescom_piece`
`journal IN ('HA','ACH','HAA','HAC','HAS')` — toutes les écritures issues d'un journal d'achat Gescom. Si tu en trouves d'autres (BAN, OD …) qui passent quand même par Gescom, complète la liste `GESCOM_JNL`. La cellule **8 (diagnostic)** sort la distribution par journal pour t'aider à auditer.

### Colonnes Lovable-owned — ne pas inclure
`note_user` et `status_user` sont gérées uniquement par l'App Task. Le notebook ne les envoie JAMAIS dans le payload → PostgREST (et donc `bulk-upsert`) ne touche pas ces colonnes lors d'un upsert. Defense in depth supplémentaire côté DB : un trigger gèle aussi ces colonnes pour tout rôle ≠ service_role (cf. migration `it_001`).

### Pattern d'envoi
Edge Function `bulk-upsert`, `conflict_key = entry_key`, `on_conflict = merge-duplicates`, batches de 500. Mêmes helpers que `Notebook_SYNC_DIVALTO_BUDGET_IT_v2`.

In [ ]:
# ─────────────────────────────────────────
# 0) Credentials Supabase
# Variables injectées automatiquement par le pipeline comme Base parameters
# Ne pas décommenter le bloc variableLibrary (mode pipeline uniquement)
# ─────────────────────────────────────────

#env = notebookutils.variableLibrary.getLibrary("env-temp")
#SUPABASE_URL       = env.SUPABASE_URL
#SYNC_SECRET        = env.SYNC_SECRET
#SUPABASE_ANON_KEY  = env.SUPABASE_PUBLISHABLE_KEY

import os, math, time, re, requests
from decimal import Decimal
from datetime import date, datetime
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, DoubleType, DateType, IntegerType, BooleanType

GOLD_DB         = "lakehouse_gold"
BULK_UPSERT_URL = f"{SUPABASE_URL.rstrip('/')}/functions/v1/bulk-upsert"
BATCH_SIZE      = 500
SLEEP_S         = 0.02
TABLE           = "supplier_accounting_entries"
CONFLICT_KEY    = "entry_key"

# Heuristique : journaux Gescom (le notebook applique aussi un OR sur la
# présence d'un FULLCDNO/FULLFANO si dispo dans C8). À auditer via la
# cellule de diagnostic.
GESCOM_JNL      = ["HA", "ACH", "HAA", "HAC", "HAS"]

# Périmètre temporel (cf. brief §9.3)
DATE_DEBUT      = "2022-01-01"

def bulk_headers():
    return {
        "Content-Type":  "application/json",
        "Authorization": f"Bearer {SUPABASE_ANON_KEY}",
        "apikey":        SUPABASE_ANON_KEY,
        "x-sync-secret": SYNC_SECRET,
    }

print("[0] Config OK")
print(f"    SUPABASE_URL    = {SUPABASE_URL}")
print(f"    SYNC_SECRET set = {bool(SYNC_SECRET)}")
print(f"    ANON_KEY set    = {bool(SUPABASE_ANON_KEY)}")
print(f"    GESCOM_JNL      = {GESCOM_JNL}")
print(f"    DATE_DEBUT      = {DATE_DEBUT}")

In [ ]:
# ─────────────────────────────────────────
# 1) Helpers sérialisation + envoi (identiques à Notebook_SYNC_DIVALTO_BUDGET_IT_v2)
# ─────────────────────────────────────────

_CTRL_RE = re.compile(r"[\x00-\x08\x0B\x0C\x0E-\x1F]")

def _clean_str(s):
    if s is None: return None
    return _CTRL_RE.sub("", str(s)).rstrip() or None

def _to_jsonable(v):
    if v is None: return None
    if isinstance(v, bool): return v
    if isinstance(v, int): return v
    if isinstance(v, float):
        return None if (math.isnan(v) or math.isinf(v)) else round(v, 2)
    if isinstance(v, Decimal):
        try:
            f = float(v); return None if (math.isnan(f) or math.isinf(f)) else round(f, 2)
        except Exception: return _clean_str(str(v))
    if isinstance(v, (date, datetime)): return v.isoformat()
    if isinstance(v, (bytes, bytearray)):
        return _clean_str(v.decode("utf-8", errors="ignore"))
    if isinstance(v, str):
        s = _clean_str(v)
        if not s or s in ("1899-12-30", "0000-00-00"): return None
        if re.match(r"^\+?\d{5,}-\d{2}-\d{2}", s): return None
        return s
    if isinstance(v, dict): return {k: _to_jsonable(val) for k, val in v.items()}
    if isinstance(v, (list, tuple)): return [_to_jsonable(x) for x in v]
    return _clean_str(str(v))

def _sanitize(rec):
    return {k: _to_jsonable(v) for k, v in rec.items()}

def _post_batch(records, on_conflict="merge-duplicates"):
    if not records: return 0, None
    payload = {
        "table":        TABLE,
        "conflict_key": CONFLICT_KEY,
        "records":      [_sanitize(r) for r in records],
        "on_conflict":  on_conflict,
    }
    try:
        r = requests.post(BULK_UPSERT_URL, headers=bulk_headers(), json=payload, timeout=180)
        if r.status_code in (200, 201, 204): return len(records), None
        return 0, f"HTTP {r.status_code}: {r.text[:300]}"
    except Exception as e:
        return 0, str(e)

def send_to_supabase(records, label):
    ok_total, err_total, errs = 0, 0, []
    for i in range(0, len(records), BATCH_SIZE):
        chunk = records[i:i+BATCH_SIZE]
        nb, err = _post_batch(chunk)
        ok_total += nb
        if err:
            err_total += len(chunk)
            errs.append(f"batch {i//BATCH_SIZE+1}: {err}")
        time.sleep(SLEEP_S)
    print(f"[{label}] ✅ {ok_total}/{len(records)} envoyés | ❌ {err_total} erreurs")
    for m in errs: print("  ⚠️", m)
    return ok_total, err_total

print("[1] Helpers OK")

In [ ]:
# ─────────────────────────────────────────
# 2) Lecture C8_gold
# Adapte les noms de colonnes si ton C8_gold n'utilise pas ces conventions :
#   dos, jnl, ecrno, ecrlg, ecrdt, compte, tiers, sens, mont/mt,
#   devise, mtdev (montant devise), libelle/lib, axe_0001..0003
# ─────────────────────────────────────────

df_c8 = spark.table(f"{GOLD_DB}.C8_gold")
print(f"[2] C8_gold chargé : {df_c8.count()} lignes brutes")
df_c8.printSchema()

In [ ]:
# ─────────────────────────────────────────
# 3) Filtre comptes F* + période + construction du DataFrame final
# ─────────────────────────────────────────

# IMPORTANT : adapte les noms de colonnes ci-dessous à TON schéma C8_gold.
# Les noms suivants sont une convention standard Divalto — vérifie via
# df_c8.printSchema() (cellule précédente) et corrige si besoin.

df_src = (
    df_c8
    # Compte auxiliaire fournisseur (F*)
    .filter(F.col("compte").startswith("F"))
    # Période
    .filter(F.col("ecrdt") >= F.lit(DATE_DEBUT).cast(DateType()))
    .filter(F.col("dos").isNotNull())
    .filter(F.col("jnl").isNotNull())
    .filter(F.col("ecrno").isNotNull())
    .filter(F.col("ecrlg").isNotNull())
)

df_entries = (
    df_src.select(
        # Clé : md5 sur (DOS|JNL|ECRNO|ECRLG)
        F.md5(F.concat_ws("|",
            F.col("dos").cast(StringType()),
            F.col("jnl").cast(StringType()),
            F.col("ecrno").cast(StringType()),
            F.col("ecrlg").cast(StringType()),
        )).alias("entry_key"),
        # ── Identifiants ──
        F.col("dos").cast(StringType()).alias("dos"),
        F.trim(F.col("jnl").cast(StringType())).alias("journal"),
        F.col("ecrno").cast(StringType()).alias("numero"),
        F.col("ecrlg").cast(IntegerType()).alias("ecrlg"),
        # ── Date ──
        F.col("ecrdt").cast(DateType()).alias("date"),
        # ── Compte + fournisseur ──
        F.trim(F.col("compte").cast(StringType())).alias("compte"),
        F.trim(F.col("compte").cast(StringType())).alias("supplier_code"),
        # Raison sociale : pas dispo en C8 seul → null en V1, à enrichir via
        # une jointure avec C3_gold (table tiers) si besoin.
        F.lit(None).cast(StringType()).alias("supplier_name"),
        # ── Libellé / montant / sens ──
        F.trim(F.col("lib").cast(StringType())).alias("libelle_ecriture"),
        F.col("mt").cast(DoubleType()).alias("montant"),
        F.col("sens").cast(IntegerType()).alias("sens"),
        F.when(F.col("sens") == 2, -F.col("mt")).otherwise(F.col("mt")).cast(DoubleType()).alias("solde"),
        # ── Devise ──
        F.coalesce(F.trim(F.col("devise").cast(StringType())), F.lit("EUR")).alias("devise"),
        F.col("mtdev").cast(DoubleType()).alias("montant_devise"),
        # ── Axes analytiques ──
        F.trim(F.col("axe_0001").cast(StringType())).alias("axe_1"),
        F.trim(F.col("axe_0002").cast(StringType())).alias("axe_2"),
        F.trim(F.col("axe_0003").cast(StringType())).alias("axe_3"),
        # ── Heuristique Gescom : journal d'achat OU pièce Gescom liée ──
        # FULLCDNO/FULLFANO ne sont pas toujours en C8 : protégé par try/coalesce.
        (
            F.upper(F.trim(F.col("jnl").cast(StringType()))).isin(GESCOM_JNL)
        ).alias("has_gescom_piece"),
        # ── Référence pièce externe (si dispo) ──
        F.trim(F.col("piref").cast(StringType())).alias("reference_externe")
          if "piref" in df_c8.columns else F.lit(None).cast(StringType()).alias("reference_externe"),
        # ── Code projet (dérivé du DOS — à raffiner via une table de mapping si besoin) ──
        F.col("dos").cast(StringType()).alias("project_code"),
        # ── Horodatage sync ──
        F.current_timestamp().alias("fabric_synced_at"),
    )
    # Dédup défensive (au cas où le même entry_key apparaîtrait deux fois)
    .dropDuplicates(["entry_key"])
)

nb = df_entries.count()
print(f"[3] Écritures à pousser : {nb}")
df_entries.show(5, truncate=False)

In [ ]:
# ─────────────────────────────────────────
# 4) Diagnostic — distribution par journal & has_gescom_piece
# À auditer : si certaines OD passent par un journal HA*, basculer
# l'heuristique sur la présence d'un FULLCDNO / lien MOUV.
# ─────────────────────────────────────────

print("── Top 20 journaux ──")
(df_entries
    .groupBy("journal", "has_gescom_piece")
    .count()
    .orderBy(F.col("count").desc())
    .show(20, truncate=False)
)

print("── Split has_gescom_piece ──")
(df_entries
    .groupBy("has_gescom_piece")
    .agg(F.count("*").alias("nb"),
         F.round(F.sum(F.abs(F.col("solde"))), 0).alias("volume_abs"))
    .show(truncate=False)
)

In [ ]:
# ─────────────────────────────────────────
# 5) Envoi vers Supabase
# IMPORTANT : on n'envoie PAS note_user / status_user — colonnes Lovable-owned.
# PostgREST ne touchera pas les colonnes absentes du payload.
# ─────────────────────────────────────────

records = [row.asDict(recursive=True) for row in df_entries.collect()]
ok, err = send_to_supabase(records, "SUPPLIER_ENTRIES")

print("")
print("=" * 60)
print("SYNC DIVALTO C8 → supplier_accounting_entries — RÉSUMÉ")
print("=" * 60)
print(f"  Écritures envoyées : {ok} / {len(records)}")
print(f"  Erreurs            : {err}")
if err == 0:
    print("  ✅ Sync OK")
else:
    print("  ⚠️  Vérifier les logs ci-dessus")
print("=" * 60)

### Points d'attention V1

1. **Noms de colonnes C8_gold** : la cellule `filter-and-build` utilise les conventions Divalto standard (`dos`, `jnl`, `ecrno`, `ecrlg`, `ecrdt`, `compte`, `mt`, `mtdev`, `sens`, `lib`, `devise`, `axe_0001..0003`). Si ton C8_gold a d'autres noms, adapte (la cellule `read-c8` imprime le schéma pour t'aider).
2. **`supplier_name`** : non rempli en V1 (pas dispo dans C8 seul). À ajouter via une jointure avec `C3_gold` (table tiers Divalto) si nécessaire — exemple :
   ```python
   df_c3 = spark.table(f"{GOLD_DB}.C3_gold").select(
       F.col("tiers").alias("compte"),
       F.trim(F.col("rs").cast(StringType())).alias("supplier_name"),
   )
   df_entries = df_entries.drop("supplier_name").join(df_c3, "compte", "left")
   ```
3. **`has_gescom_piece`** : heuristique sur le journal. Audite via la cellule `diag` — si tu vois des OD passant par un journal HA*, complète/modifie la liste `GESCOM_JNL` ou bascule sur la présence d'un `FULLCDNO`.
4. **`project_code`** : actuellement = `dos` brut. Si tu veux un mapping DOS → code projet (KEON / NASK / …), ajoute une table de référence et joins-là ici.
5. **Idempotence** : `entry_key` est stable (md5 sur 4 colonnes immuables), donc le notebook est relançable sans doublon.
6. **Préservation des colonnes Lovable-owned** : le payload n'inclut ni `note_user` ni `status_user` ; PostgREST ne touche que les colonnes envoyées. La migration `it_001` pose en plus un trigger qui gèle les colonnes Fabric-owned pour tout rôle ≠ service_role (defense in depth).